In [17]:
import pandas as pd
import matplotlib.pyplot as plt
import torch
from collections import Counter
import copy

In [ ]:
df = pd.read_csv(f"../../../../data/dutch/dutch_census.csv")

In [ ]:
df.head()

In [ ]:
df["edu_level"].value_counts()

In [ ]:
df_edu_level = df[df["edu_level"] != 0]
print(df_edu_level.shape)
df_edu_level.to_csv(f"../../../../data/dutch/dutch_census_edu_level.csv")

In [ ]:
df.shape

In [ ]:
nodes = list(range(50))

In [ ]:
datasets = {}
for node in nodes:
    datasets[node] = torch.load(f"../../../../data/dutch/federated_2/{node}/train.pt")

In [ ]:
education_levels = {}
for node in nodes:
    education_levels[node] = Counter(datasets[node].sensitive_features)

In [ ]:
# mapping = {"0.0": 0, "0.2": 1, "0.4": 2, "0.6": 3, "0.8": 4, "1.0": 5}
gender = {}
total = 0
targets_ = {}
for node in nodes:
    values = []
    targets = []
    for sample in datasets[node].samples:
        values.append(sample[10])
    for target in datasets[node].targets:
        targets.append(target)
    gender_target = zip(values, targets)

    values = Counter([str(item) for item in values])
    gender[node] = values
    targets_[node] = []
    for item in gender_target:
        targets_[node].append(item)
for node in nodes:
    targets_[node] = Counter(targets_[node])

In [ ]:
def plot_distribution(counters):
    import matplotlib.pyplot as plt

    from collections import Counter

    for key, counter in counters.items():
        first_key = key
        break
    # Extracting data for plotting
    keys = sorted(counters[first_key].keys())  # Sorting keys for consistent ordering
    possible_keys = []
    for item in counters:
        possible_keys.append(list(counters[item].keys()))
    keys = sorted(list(set([item for sublist in possible_keys for item in sublist])))
    counter_values = [[counter[key] for key in keys] for counter in counters.values()]

    counter_keys = [[key for key in counter.keys()] for counter in counters.values()]
    # Plotting
    plt.figure(figsize=(15, 8))
    bottom_sum = [0] * len(counters)

    for key in keys:
        values = []
        print(key)
        for _, count in counters.items():
            values.append(count[key])
        plt.bar(
            range(len(values)),
            values,
            bottom=bottom_sum,
            label=str(key),
        )
        bottom_sum = [sum(x) for x in zip(bottom_sum, values)]

    plt.xlabel("Keys")
    plt.ylabel("Counts")
    plt.title("Distribution of Counters")
    plt.legend()
    plt.show()

In [ ]:
plot_distribution(education_levels)

In [ ]:
plot_distribution(targets_)

In [ ]:
test_set = ["0", "1", "2", "3", "4", "25", "26", "27", "28", "29"]
training_set = [
    "17",
    "9",
    "12",
    "6",
    "16",
    "24",
    "5",
    "8",
    "11",
    "21",
    "20",
    "10",
    "13",
    "22",
    "14",
    "47",
    "31",
    "40",
    "35",
    "48",
    "33",
    "43",
    "32",
    "44",
    "42",
    "30",
    "46",
    "49",
    "38",
    "39",
]
validation_set = ["7", "18", "15", "23", "19", "36", "37", "41", "45", "34"]

In [ ]:
test_set_targets = {int(node): targets_[int(node)] for node in test_set}
training_set_targets = {int(node): targets_[int(node)] for node in training_set}
validation_set_targets = {int(node): targets_[int(node)] for node in validation_set}

plot_distribution(test_set_targets)
plot_distribution(training_set_targets)
plot_distribution(validation_set_targets)

In [ ]:
test_education_levels = {int(node): education_levels[int(node)] for node in test_set}
training_education_levels = {int(node): education_levels[int(node)] for node in training_set}
validation_education_levels = {int(node): education_levels[int(node)] for node in validation_set}

plot_distribution(test_education_levels)
plot_distribution(training_education_levels)
plot_distribution(validation_education_levels)

# Poisoning Dutch 

In [23]:
df = pd.read_csv(f"../../../../data/dutch/dutch_census_edu_level.csv")

In [24]:
for i in range(5):
    print(
        f"edu_level {i}, occupation=1",
        df[(df["edu_level"] == i) & (df["occupation"] == 21)].shape[0],
    )
    print(
        f"edu_level {i}, occupation=0",
        df[(df["edu_level"] == i) & (df["occupation"] == 549)].shape[0],
    )

edu_level 0, occupation=1 0
edu_level 0, occupation=0 0
edu_level 1, occupation=1 732
edu_level 1, occupation=0 3781
edu_level 2, occupation=1 2267
edu_level 2, occupation=0 10059
edu_level 3, occupation=1 8051
edu_level 3, occupation=0 14621
edu_level 4, occupation=1 1466
edu_level 4, occupation=0 1114


In [25]:
sensitive_value = "edu_level"
target = "occupation"
# label flipping Gender=1, Target value: Smiling:1 to Smiling:-1 in the 50% of samples
df.loc[df[(df[sensitive_value] == 4) & (df[target] == 21)].sample(frac=0.25).index, target] = 549

df.loc[df[(df[sensitive_value] == 4) & (df[target] == 549)].sample(frac=0.25).index, target] = 21

In [26]:
for i in range(5):
    print(
        f"edu_level {i}, occupation=1",
        df[(df["edu_level"] == i) & (df["occupation"] == 21)].shape[0],
    )
    print(
        f"edu_level {i}, occupation=0",
        df[(df["edu_level"] == i) & (df["occupation"] == 549)].shape[0],
    )

edu_level 0, occupation=1 0
edu_level 0, occupation=0 0
edu_level 1, occupation=1 732
edu_level 1, occupation=0 3781
edu_level 2, occupation=1 2267
edu_level 2, occupation=0 10059
edu_level 3, occupation=1 8051
edu_level 3, occupation=0 14621
edu_level 4, occupation=1 1470
edu_level 4, occupation=0 1110


In [27]:
df.to_csv(f"../../../../data/dutch/dutch_census_edu_level_poisoned.csv")

In [20]:
import wandb
import pandas as pd
import numpy as np

api = wandb.Api()
run = api.run("/lucacorbucci/income_journal/runs/9ivi76a7")


print(run.scan_history())

In [24]:
# check values that are not nan
a = np.array([item for item in list(df["Test Group Error Rate 4"]) if isinstance(item, float) and item > 0])
b = np.array([item for item in list(df["Test Group Error Rate 8"]) if isinstance(item, float) and item > 0])
a - b

array([0.18460196, 0.23759854, 0.34165357, 0.29979886, 0.3797933 ,
       0.3519631 , 0.28184921, 0.28394851, 0.29673186, 0.37413457])

In [7]:
df = pd.DataFrame(run.scan_history())
df.head()

,FL Round,Test Group Accuracy 4,Test Group Accuracy 5,Train Group Error Rate 6,Test Group Error Rate 9,Test Group Accuracy 9,Train Group Accuracy 5,Error Rate Client 17 After Local train,Error Rate Client 15 After Local train,Test Node 3 - Acc.,...,Test Node 0 - Acc.,Test Node 6 - Acc.,Test Node 2 - Error Rate (Softmax),Error Rate Client 40 After Local train,Test Node 7 - Acc.,Test Node 1 - Acc.,Train Accuracy,Train Group Accuracy 1,Test Node 1 - Error Rate (Softmax),Test Max Error Rate Client 3 - (Argmax)
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.765636,NaN,NaN,NaN
1,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.747012,NaN,NaN
2,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
